# ETL Load Notebook

This notebook loads the cleaned and KPI-ready datasets into a new SQLite database named clean_data. The resulting database can be used for analytics, reporting, or further integration into BI tools.

## What this notebook does
- Creates a new SQLite database file.
- Loads the transformed datasets into database tables.
- Validates that the tables were created successfully.
- Prints row counts and table names for verification.

In [ ]:
# Cell 1 — Prepare the target database connection
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine

DB_PATH = Path("ETL/clean_data.db")
PROCESSED_DIR = Path("ETL/ETL_processed_data")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

engine = create_engine(f"sqlite:///{DB_PATH}")

customers = pd.read_csv("ETL/staging/customers.csv")
products = pd.read_csv("ETL/staging/products.csv")
orders = pd.read_csv(PROCESSED_DIR / "orders_enriched.csv")
feedback = pd.read_csv(PROCESSED_DIR / "feedback_clean.csv")
kpis = pd.read_csv(PROCESSED_DIR / "kpis.csv")

print("Target database path:", DB_PATH)
print("Loaded processed files:", ["orders_enriched.csv", "feedback_clean.csv", "kpis.csv"])

In [ ]:
# Cell 2 — Load the transformed datasets into SQLite tables
customers.to_sql("customer_dim", engine, if_exists="replace", index=False)
products.to_sql("product_dim", engine, if_exists="replace", index=False)
orders.to_sql("orders_clean", engine, if_exists="replace", index=False)
feedback.to_sql("feedback_clean", engine, if_exists="replace", index=False)
kpis.to_sql("kpi_summary", engine, if_exists="replace", index=False)

print("Load completed successfully")
print("Tables created in the database:", ["customer_dim", "product_dim", "orders_clean", "feedback_clean", "kpi_summary"])

In [ ]:
# Cell 3 — Validate the database contents
with engine.connect() as connection:
    tables = connection.exec_driver_sql("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    print("Database tables:", tables)
    row_counts = {table: connection.exec_driver_sql(f"SELECT COUNT(*) FROM {table}").scalar() for table, in tables}
    print("Row counts:", row_counts)